# COCO100 — reusable embedding bank

Prepare Native and PSNR39/42/45 PNGs once, then reuse them in all three channels. No attacks, training, calibration or generation-time methods run here.

One image-specific 30-bit message is shared by all methods and modes. Start with 5 images, inspect the report, then change only N_IMAGES to 100. All 32 requested methods are listed, but missing weights, MATLAB and the unimplemented Camera-Shooting Instance adapter remain explicit blockers.

**Colab:** use an NVIDIA GPU and high-RAM runtime if available. ChunkySeal is particularly demanding; no GPU is a compatibility guarantee. Results persist on Drive; cloud Colab cannot directly access your E: SSD.

**Windows:** open this notebook in Jupyter, connect E:, and supply the existing COCO100 originals. Some upstream model environments are Linux-specific. The CPU checks do not establish compatibility of all 32 methods with Windows.

## 1. Settings

In [ ]:
# Change paths here. Start at 5; after inspecting the report, set 100.
from pathlib import Path
import sys, os, json, shutil, subprocess, importlib.util
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

N_IMAGES = 5                         # 100 for the complete image bank
MODELS = "all"                       # All 32 requested catalogue entries
DEVICE = "cuda"                      # "cpu" only for a CPU-capable local test
IDENTITY_CHECK = True                # Decode saved PNGs once; no attacks

if IN_COLAB:
    OUTPUT_ROOT = Path("/content/drive/MyDrive/watermarking_embedding_bank_v1")
    SCRATCH_ROOT = Path("/content/watermarking_embedding_scratch")
    COCO_ROOT = Path("/content/drive/MyDrive/watermarking_coco100_visible/coco_fixed_512")
else:
    OUTPUT_ROOT = Path("E:/watermarking_embedding_bank_v1")
    SCRATCH_ROOT = Path.home() / "wmbench_embedding_scratch"
    COCO_ROOT = Path("E:/watermarking_inputs/coco_fixed_512")

MANUAL_ASSETS_ROOT = OUTPUT_ROOT / "_manual_assets"
SETTINGS = {
    "output_root": str(OUTPUT_ROOT), "scratch_root": str(SCRATCH_ROOT),
    "coco_root": str(COCO_ROOT), "manual_assets_root": str(MANUAL_ASSETS_ROOT),
    "n_images": N_IMAGES, "models": MODELS, "device": DEVICE,
    "identity_check": IDENTITY_CHECK, "cleanup": True,
    "max_hours": 12, "prepare_minutes": 60, "model_minutes": 180,
    "min_free_disk_gib": 20, "min_free_ram_gib": 2, "max_slot_gib": 150,
}
print("Output:", OUTPUT_ROOT)
print("Originals:", COCO_ROOT / "test", "and", COCO_ROOT / "manifest.csv")
print("Images per model:", N_IMAGES, "| One 30-bit message per image | Four modes")


## 2. Source bundle and lightweight controller

This is source code, not an executable checkpoint bundle. Each model gets its own Python environment. Preserve the exact originals and manifest from the earlier runs.

In [ ]:
# Source bundle only: model weights are downloaded one method at a time.
import base64, hashlib, io, zipfile, sysconfig
if IN_COLAB:
    drive.mount("/content/drive")
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive is not mounted; do not save results to ephemeral SSD")
elif not OUTPUT_ROOT.anchor or not Path(OUTPUT_ROOT.anchor).exists():
    raise FileNotFoundError("Connect the Samsung SSD or change OUTPUT_ROOT")
if DEVICE.startswith("cuda") and not shutil.which("nvidia-smi"):
    raise RuntimeError("NVIDIA GPU/driver not found. Select a GPU runtime in Colab, or use DEVICE='cpu' for a CPU-capable local test.")
if not shutil.which("git"):
    raise RuntimeError("Git is required for pinned model sources. Install Git and restart Jupyter.")

SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)
PAYLOAD_SHA256 = "9736c8bfb5a32d59c9f622fd61eff2537bb02a623107c96870a3d78277897aac"
PAYLOAD_BASE64 = "..."
payload = base64.b64decode(PAYLOAD_BASE64)
if hashlib.sha256(payload).hexdigest() != PAYLOAD_SHA256:
    raise ValueError("Embedded source bundle checksum mismatch")
PROJECT = SCRATCH_ROOT / "source" / PAYLOAD_SHA256[:16]
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for member in archive.infolist():
        target = (PROJECT / member.filename).resolve()
        if not target.is_relative_to(PROJECT.resolve()):
            raise ValueError("Unsafe source archive path")
        content = archive.read(member)
        if target.exists() and target.read_bytes() != content:
            raise ValueError(f"Modified release source: {target}; use a fresh source directory")
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)

# The kernel never receives Torch or incompatible model dependencies.
try:
    import pip
except ImportError:
    subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.8.15"], check=True)
import uv
UV = str(uv.find_uv_bin())
controller_dir = SCRATCH_ROOT / "controller"
CONTROLLER = controller_dir / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not CONTROLLER.exists():
    subprocess.run([UV, "venv", "--python", "3.10", str(controller_dir)], check=True)
subprocess.run([UV, "pip", "install", "--python", str(CONTROLLER),
    "numpy==1.26.4", "Pillow==10.4.0", "psutil==6.1.1"], check=True)
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(PROJECT / "src")
ENV["PATH"] = str(Path(UV).parent) + os.pathsep + ENV.get("PATH", "")
ENV["PYTHONUNBUFFERED"] = "1"
CONFIG = SCRATCH_ROOT / "embedding_bank_settings.json"
CONFIG.write_text(json.dumps(SETTINGS, indent=2), encoding="utf-8")
if not (OUTPUT_ROOT / "_inputs/manifest.json").is_file():
    if not (COCO_ROOT / "test").is_dir() or not (COCO_ROOT / "manifest.csv").is_file():
        raise FileNotFoundError(f"Copy the existing exact COCO100 test/ + manifest.csv to {COCO_ROOT}. No new dataset will be substituted.")
print("Isolated controller:", CONTROLLER)
print("Free scratch SSD: %.1f GiB" % (shutil.disk_usage(SCRATCH_ROOT).free / 1024**3))
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
print("Ready. Saved images will survive model-environment cleanup.")


## 3. Embed, match, save and verify

Completed images are reused by hash. Only pending work is computed. After model processing, disposable environments/weights are deleted; the bank is not. Re-running may reinstall weights but will not re-embed completed images. A technical decode failure is retryable; a measured poor BER is not re-sampled.

In [ ]:
# Re-running resumes verified images, including after a Colab disconnect.
# Stop an older run before starting this cell in another notebook/runtime.
command = [str(CONTROLLER), "-m", "wmbench.embedding_bank", "run", "--config", str(CONFIG)]
process = subprocess.Popen(command, env=ENV, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f"Controller exited with code {code}. Saved records remain; inspect the logs.")
except BaseException:
    process.terminate()
    process.wait(timeout=30)
    raise
print("Review:", OUTPUT_ROOT / "_reports/REPORT.md")
print("A partial/blocked report is not a successful 32-model bank.")


## 4. Coverage report and SSD transfer

In [ ]:
# Refresh from saved files; does not reload models or run inference.
subprocess.run([str(CONTROLLER), "-m", "wmbench.embedding_bank", "report",
    "--config", str(CONFIG)], env=ENV, check=True)
from IPython.display import Markdown, display
display(Markdown((OUTPUT_ROOT / "_reports/REPORT.md").read_text(encoding="utf-8")))
print("Transfer the WHOLE output folder to E:/watermarking_embedding_bank_v1, not only its PNGs.")
print("No large output ZIP is created automatically; this avoids duplicate disk usage.")


## Protocol and limitations

# Reusable COCO embedding bank

This stage prepares watermarked images once. The same saved PNGs can later be
used for digital attacks, neural transformations and screen–camera experiments.
It does **not** run attacks, generate calibration images, train models or claim
that the complete benchmark has finished.

The requested catalogue contains **32 post-hoc methods and variants**. Inclusion
in the catalogue is not proof that the required weights, software or adapter are
available. Missing prerequisites are reported explicitly; no other method or
checkpoint is substituted to fill a folder.

## 1. Inputs and messages

- Use the fixed historical COCO100 corpus: RGB PNGs, **512 × 512**, with its
  original manifest and image hashes. Do not reconstruct missing originals from
  newly downloaded JPEGs and treat them as the same corpus.
- `N_IMAGES = 100` prepares the full bank. Start with `N_IMAGES = 5` to check the
  complete embedding workflow on the fixed first five images.
- Assign **one deterministic, balanced 30-bit message per image**: 15 zeros and
  15 ones. Store the assignment; use the same message for that image across all
  methods and modes. This is the newly agreed one-message design, not the older
  five-messages-per-image experiment.
- Where a model accepts more than 30 bits, use the registered common-bit
  positions and deterministic filler. Preserve the complete native payload,
  common positions and any reserved bits in metadata. Do not overwrite
  method-specific detection positions or silently truncate a message.
- Record raw capacity, useful payload length and ECC separately. Fang retains
  its published CRC/BCH coding; it is not a raw30 neural watermark.

The five-image run is a prefix of the 100-image run. Changing only `N_IMAGES`
from 5 to 100 must not change the original files, messages or scientific
configuration of the first five images. All 100 originals and their manifest
are frozen when the bank is created, even for a five-image run.

## 2. Embedding and quality modes

For each method–image pair, embed once and save the **Native RGB uint8 PNG**.
Construct the three matched variants from that saved Native image, not from a
second stochastic embedding call or an unsaved floating-point tensor.

| Folder | Rule |
| --- | --- |
| `native/` | Configured method and checkpoint; no external strength matching |
| `matched_39/` | Target PSNR 39 dB, tolerance ±0.05 dB |
| `matched_42/` | Target PSNR 42 dB, tolerance ±0.05 dB |
| `matched_45/` | Target PSNR 45 dB, tolerance ±0.05 dB |

For original pixels `x` and saved Native pixels `y`, both in `[0,255]`:

```text
matched(alpha) = round_ties_to_even(clip(x + alpha * (y - x), 0, 255))
0 <= alpha <= 4
PSNR = 10 * log10(255^2 / mean((x - matched(alpha))^2))
```

Use **one scalar alpha for the whole image**. There is no per-pixel correction,
decoder-guided search, seed selection or message substitution. Check the achieved
PSNR after PNG save/reload. Keep the measured PSNR, alpha and matching status.

If a target cannot be reached within the fixed search policy and tolerance,
record it as unavailable. Do not publish the nearest image as a successful
matched result. Native can still be valid even when a matched mode is unavailable
or clean message extraction is imperfect. Record clean extraction separately;
do not keep trying until a desirable bit accuracy appears.

The optional `identity_check` decodes saved images without an attack and is
enabled by default. Disabling it skips an important integration check; it does
not change embedding or matching and must not be described as verified clean
recovery.

A technical decoding exception is retried on the next run using the same fixed
seed, without re-embedding the Native image. The previous diagnostic is archived.
A successfully measured nonzero BER is never retried to obtain a better message.

## 3. Strict method allowlist

Only these 32 entries belong to this bank:

| Group | Methods / variants |
| --- | --- |
| Existing digital baselines (15) | TrustMark-Q, WAM, VideoSeal, PixelSeal, ChunkySeal, VINE-B, VINE-R, RoSteALS, InvisMark, RivaGAN benchmark, DWT-DCT-SVD, MBRS, CIN, HiDDeN benchmark, EditGuard |
| Additional post-hoc methods (6) | ARWGAN, SSL Watermarking multibit, MaskWM, Robust-Wide, SepMark, OmniGuard |
| Physical specialists (6) | StegaStamp ONNX, PIMoG, SSRH, SepWater, Fang Screen, Camera-Shooting Instance |
| New additions (5) | TrustMark-B, TrustMark-P, FIN, FNNS from OmniSealBench, DistSeal post-hoc DC-AE |

TrustMark variants and benchmark-specific implementations retain their explicit
labels. The nine generation-time methods are outside this stage. Previously
removed methods are not brought back through an old registry's default list.

### Known prerequisites and verification limits

| Entry | Requirement or limitation |
| --- | --- |
| StegaStamp ONNX | Uses the WIBE raw100 ONNX conversion, not a claim of numerical equivalence to every TensorFlow implementation. CUDA inference requires the compatible ONNX Runtime libraries. |
| PIMoG / SSRH | The original networks operate at 128 pixels in BGR. The documented adapter lifts their residual to the 512-pixel carrier. This adaptation stays fixed in all modes. |
| SepWater | Supply the full matching author checkpoint; required encoder, autoencoder and decoder tensors must exist. A missing checkpoint is a blocker. |
| Fang Screen | Requires licensed MATLAB with Image Processing and Communications Toolboxes. MATLAB Runtime or Octave is not a verified substitute. Feature-poor images can fail the original embedding algorithm. |
| Camera-Shooting Instance | The current inventory has no verified runnable embedding/payload contract. It remains blocked until integration and checkpoint validation are completed. |
| EditGuard / OmniGuard | Require the correct copyright-watermark checkpoints; localization-only weights are not interchangeable. |
| New additions | Public source and checkpoints do not replace a canary of the exact selected implementation, payload mapping and target environment. |

Historical real CPU interface checks exist for StegaStamp ONNX, PIMoG and SSRH on
one author-provided image. They do not establish COCO100 coverage or validate the
production GPU environments. Technical failures are neither successful outputs
nor evidence that an algorithm cannot carry the message.

## 4. Output layout

The method is the top-level image folder, followed by the quality mode:

```text
watermarking_embedding_bank_v1/
├── _inputs/                 # Exact originals, manifest and message assignment
│   └── images/              # <12-digit COCO ID>.png
├── bank.json                # Frozen release, model registry and scientific policy
├── _reports/                # Coverage tables and report
├── trustmark_q/
│   ├── native/
│   ├── matched_39/
│   ├── matched_42/
│   ├── matched_45/
│   └── _meta/               # records/, keys/, executions/, logs/
├── wam/
│   ├── native/
│   ├── matched_39/
│   ├── matched_42/
│   └── matched_45/
└── ...
```

Image names identify their COCO source. Metadata links each file to its original,
message, adapter sidecar, model revision, checkpoint hashes and measured quality.
An empty or incomplete mode folder is not a successful result; consult the
coverage report. Per-method records are stored under
`_meta/records/<image_id>_<mode>.json`; decoder sidecars are under
`_meta/keys/<image_id>/metadata.json`. Execution records and logs are retained
alongside them. Start with `_reports/REPORT.md` and its accompanying CSV tables
when reviewing a run.

With 100 images, one message and all four modes available, the upper count is
**12,800 marked PNGs**. At five images it is **640 PNGs**. Model weights, Python
environments and download caches are separate from this persistent image bank.
Size the output and scratch disks separately, using the five-image run to measure
actual file sizes.

## 5. Windows or Colab

### Windows with the Samsung SSD

Use a dedicated directory, not the drive root:

```text
OUTPUT_ROOT = E:/watermarking_embedding_bank_v1
```

Set `SCRATCH_ROOT` to a separate local working directory with sufficient free
space for the active model's source, environment and weights. Persistent results
belong under `OUTPUT_ROOT`; temporary installations do not need to be copied
into the image folders. Keep the SSD connected during writing.

Native Windows is not a blanket compatibility guarantee for all upstream code.
Some WIBE integrations use symbolic links or dependencies requiring compilation;
Windows permissions, toolchains and available GPU wheels must be checked. Fang
can use a locally installed licensed MATLAB. Unsupported environments must fail
with a reason rather than fall back silently to another implementation.

### Colab Pro+

Colab with an NVIDIA GPU is the preferred first target for the Python model
environments. Mount Drive and use:

```text
OUTPUT_ROOT = /content/drive/MyDrive/watermarking_embedding_bank_v1
SCRATCH_ROOT = /content/watermarking_embedding_scratch
```

A cloud Colab runtime **cannot write directly to the Samsung SSD connected to
the Windows computer**. Save the durable bank to Drive, then copy it to
`E:/watermarking_embedding_bank_v1`. Local `/content` storage is temporary and
must not be the only location of completed results. Colab does not supply a
licensed MATLAB installation for Fang.

Run from the project environment after preparing the input corpus and settings:

```bash
python -m pip install uv==0.8.15 .
python -m wmbench.embedding_bank run --config settings.json
```

Example `settings.json` for a first local canary; replace the input and scratch
paths with real directories. `coco_root` points to the existing corpus root with
`test/` and `manifest.csv`, not a newly selected COCO subset.

```json
{
  "output_root": "E:/watermarking_embedding_bank_v1",
  "scratch_root": "D:/watermarking_embedding_scratch",
  "coco_root": "D:/watermarking_inputs/coco100",
  "manual_assets_root": "D:/watermarking_inputs/manual_assets",
  "n_images": 5,
  "models": "all",
  "device": "cuda",
  "identity_check": true,
  "cleanup": true,
  "max_hours": 12,
  "prepare_minutes": 60,
  "model_minutes": 180,
  "min_free_disk_gib": 20,
  "min_free_ram_gib": 2,
  "max_slot_gib": 150
}
```

Use `device: "cpu"` for a CPU-capable local canary; not every model is guaranteed
to support CPU inference. `models` can be an explicit list of allowlisted IDs
for diagnosis, but a subset run is not a completed 32-method bank. Time, RAM and
disk limits are execution safeguards, not reductions of the scientific task.
With `cleanup` enabled, the controller can reclaim its managed scratch slot
between models; the persistent bank is not a cleanup target.

The notebook exposes the same operation with editable paths and `N_IMAGES`
(the JSON key is `n_images`).
Begin at five images, inspect coverage and errors, resolve prerequisites, then
increase to 100 without changing the scientific settings.

## 6. Execution, resume and transfer

- Process methods sequentially in isolated environments. A worker holds only
  its own model; incompatible model dependencies are not installed together in
  the notebook kernel.
- Commit records per image and mode. Resume checks the specification and file
  hashes, not merely whether a PNG filename exists. A partially written or
  corrupted file is not complete.
- Valid saved Native images are reused when producing or resuming matched
  modes. A stopped job does not require rerunning the entire model.
- Runtime and source provenance are recorded. Moving files for storage is not
  permission to silently merge newly computed outputs from changed checkpoints,
  payload mappings or numerical settings.
- Resume is pinned to this source release. Changing source code or the scientific
  policy requires a new bank or an explicitly audited migration. Changing only
  the image count, storage paths or runtime does not change saved PNG identities.
  Checkpoint hashes are frozen per method; new runtime/library details remain
  separate execution records. Cross-hardware numerical equivalence is not assumed.
- Logs and the final report distinguish completed, failed, blocked and
  unreachable matched jobs. The expected denominator remains the requested
  allowlist × images × modes; missing prerequisites do not turn a partial bank
  into an all-32 success.
- Transfer the **entire bank directory**, including metadata, hidden files,
  keys, originals, messages and reports. The PNG folders alone are insufficient
  for reproducible extraction. Keep secret keys out of public research releases
  unless sharing them is an explicit part of the release design.

This stage ends with a reusable, audited input bank and an honest completeness
report. Channel-specific experiments consume these saved images independently;
one channel's attacked output is not the next channel's input.
